In [1]:
import pandas as pd
import os
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Define the required columns for feature selection
req_cols = [' Packet Length Std', ' Total Length of Bwd Packets', ' Subflow Bwd Bytes', ' Destination Port', 
            ' Packet Length Variance', ' Bwd Packet Length Mean', ' Avg Bwd Segment Size', 'Bwd Packet Length Max', 
            ' Init_Win_bytes_backward', 'Total Length of Fwd Packets', ' Subflow Fwd Bytes', 'Init_Win_bytes_forward', 
            ' Average Packet Size', ' Packet Length Mean', ' Max Packet Length',' Label']

# Load the data from csv files
fraction = 0.1
frames = []

for filename in ['Wednesday-workingHours.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 
                 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
                 'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 
                 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv']:
    path = f'cicids_db/{filename}'
    if os.path.exists(path):
        print(f'Loading {filename}...')
        df_tmp = pd.read_csv(path, usecols=req_cols, encoding='cp1252').sample(frac=fraction)
        frames.append(df_tmp)

df = pd.concat(frames, ignore_index=True)

# CRITICAL: Strip leading/trailing spaces from all column names
df.columns = df.columns.str.strip()

# Label consolidation
df['Label'] = df['Label'].replace({
    'DoS GoldenEye': 'Dos/Ddos', 'DoS Hulk': 'Dos/Ddos', 'DoS Slowhttptest': 'Dos/Ddos', 
    'DoS slowloris': 'Dos/Ddos', 'Heartbleed': 'Dos/Ddos', 'DDoS': 'Dos/Ddos',
    'FTP-Patator': 'Brute Force', 'SSH-Patator': 'Brute Force',
    'Web Attack - Brute Force': 'Web Attack', 'Web Attack - Sql Injection': 'Web Attack', 
    'Web Attack - XSS': 'Web Attack'
})

df = df.fillna(0)

# Separate Features and Labels
X = df.drop(columns=['Label'])
y = df['Label'].astype(str)

# Ensure all features are numeric and non-negative for chi2
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# Max-Normalization
for col in X.columns:
    max_val = abs(X[col].max())
    if max_val != 0:
        X[col] = X[col] / max_val

# Re-scale specific columns as in the original research script
X['Init_Win_bytes_forward'] = (X['Init_Win_bytes_forward'] + 1)
X['Init_Win_bytes_backward'] = (X['Init_Win_bytes_backward'] + 1)
X['Init_Win_bytes_forward'] = X['Init_Win_bytes_forward'] / X['Init_Win_bytes_forward'].max()
X['Init_Win_bytes_backward'] = X['Init_Win_bytes_backward'] / X['Init_Win_bytes_backward'].max()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Perform chi-square feature selection
chi_selector = SelectKBest(chi2, k=10)
X_train_chi = chi_selector.fit_transform(X_train, y_train)
X_test_chi = chi_selector.transform(X_test)


Loading Wednesday-workingHours.pcap_ISCX.csv...
Loading Tuesday-WorkingHours.pcap_ISCX.csv...
Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_14652\3349442273.py:24: DtypeWarning: Columns (84) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv(path, usecols=req_cols, encoding='cp1252').sample(frac=fraction)


Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
Loading Monday-WorkingHours.pcap_ISCX.csv...
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv...
Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...


In [2]:

# get the indices of the selected features
selected_indices = chi_selector.get_support(indices=True)

# get the names of the selected features
selected_features = list(X_train.columns[selected_indices])

# print the selected features
print('Selected Features:', selected_features)

Selected Features: ['Destination Port', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'Average Packet Size', 'Avg Bwd Segment Size', 'Init_Win_bytes_forward']
